<a href="https://colab.research.google.com/github/babi00/ai4biological-pattern/blob/clean-barbara/barbara_new/explainability_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Explainability del classificatore di specie invasive. Qui si utilizzano le tecniche: Integrated Gradients, nello specifico la funzione Integrated Gradients della libreria captum; SHAP, nello specifico la funzione GradientShap della libreria captum; LIME, nello specifico la funzione lime_image della libreria lime.

Si può specificare il numero di samples da analizzare con ```num_images``` e si può filtrare per label desiderato (```target_label_name```) e specie desiderata (```species```).



In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, multilabel_confusion_matrix, confusion_matrix, ConfusionMatrixDisplay, f1_score
import seaborn as sns
import requests
from io import BytesIO
from google.colab import drive
import time
from tqdm import tqdm
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

!pip install opencv-python==4.5.1.48
import cv2

!pip install open_clip_torch
!pip install captum

from captum.attr import IntegratedGradients
import open_clip
import random
import math
from collections import Counter
import torch.nn.functional as F
from captum.attr import GradientShap

!pip install lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
from skimage.color import label2rgb
import torchvision.transforms as T
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase
import matplotlib.cm as cm





torch.manual_seed(42)
np.random.seed(42)

drive.mount('/content/drive')

# Clone the repository and checkout the 'clean-barbara' branch
!git clone --branch clean-barbara https://github.com/babi00/ai4biological-pattern.git
%cd ai4biological-pattern

# Enable sparse checkout
!git sparse-checkout init --cone

  Using cached opencv-python-4.5.1.48.tar.gz (88.3 MB)
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'ai4biological-pattern' already exists and is not an empty directory.
/content/ai4biological-pattern


In [2]:
class InvasiveSpeciesDataset(Dataset):
    def __init__(self, root_dir, transform=None, get_label_fn=None):
        self.root_dir = root_dir
        self.transform = transform
        self.entries = []
        self.label_map = {
            "native": 0,
            "introduced non-invasive": 1,
            "invasive": 2
        }

        taxa_folders = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) and d != "metadata"]
        for taxon in taxa_folders:
            meta_path = os.path.join(root_dir, "filtered_metadata", f"{taxon}_metadata.csv")
            if not os.path.exists(meta_path):
                continue
            metadata_df = pd.read_csv(meta_path)
            for idx, row in metadata_df.iterrows():
                self.entries.append((taxon, row))  # Just index, don’t open anything

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        taxon, row = self.entries[idx]
        filename = str(row["filename"])
        image_path = os.path.join(self.root_dir, taxon, filename)
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.label_map.get(row["label"])
        return image, label, idx

In [3]:
#2.Define the embedding extractor:
#ResNet without the last FC layer
def get_resnet_embeddings_extractor_model():
    #Use a pre-trained ResNet18 model
    model = models.resnet18(weights='IMAGENET1K_V1')

    #Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    embeddings_extractor= nn.Sequential(*list(model.children())[:-1]) #removes the last FC layer
    print("Model obtained...")

    return embeddings_extractor

#BioCLIP model
def get_bioclip_embeddings_extractor_model(device, bioclip_version=2):
    if bioclip_version==2:
        print("BioCLIP 2!")
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
    else:
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip')
    print("Model obtained...")
    model.to(device)
    model.eval()
    return model, preprocess

In [4]:
#•create a dummy image to take the embedding dimensions
def get_embedding_dim(embedding_model, preprocess, device):
    dummy_image = Image.new('RGB', (224, 224), color='white')
    image_tensor = preprocess(dummy_image).unsqueeze(0).to(device)
    with torch.no_grad():
        if hasattr(embedding_model, 'encode_image'):
            embedding = embedding_model.encode_image(image_tensor)
        else:
            embedding = embedding_model(image_tensor)

    embedding_dim = embedding.shape[1]
    return embedding_dim

In [5]:
#Classifier
class InvasiveSpeciesClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=256, num_classes=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(), #consider adding a Dropout layer
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

In [6]:
class FullModel(nn.Module):
    def __init__(self, embedding_model, classifier_head):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier_head = classifier_head

    def forward(self, x):
        if hasattr(self.embedding_model, 'encode_image'):
            features = self.embedding_model.encode_image(x)
        else:
            features = self.embedding_model(x)

        if features.ndim == 4:
            features = features.view(features.size(0), -1)

        return self.classifier_head(features)

In [7]:
def visualize_shap(input_tensor, model, pred_class, device):

    grad_shap = GradientShap(model)

    # baseline: random noise, or black image
    baseline = torch.zeros_like(input_tensor)

    attributions = grad_shap.attribute(input_tensor, baselines=baseline, target=pred_class)

    # Convert to numpy for visualization
    attr = attributions.squeeze().detach().cpu().numpy()
    attr = np.transpose(attr, (1, 2, 0))  # CHW → HWC
    attr = np.abs(attr).mean(axis=-1)    # grayscale

    return get_attr_map(input_tensor, attr, "Gradient SHAP Attribution")

In [25]:
def visualize_lime(image_tensor, model, pred_class, device):
    def tensor_to_numpy(img_tensor):
        img = img_tensor.cpu().numpy()
        mean = np.array([0.48145466, 0.4578275, 0.40821073])[:, None, None]
        std = np.array([0.26862954, 0.26130258, 0.27577711])[:, None, None]
        img = img * std + mean
        img = np.clip(img, 0, 1)
        img = np.transpose(img, (1, 2, 0))
        img = (img * 255).astype(np.uint8)
        return img

    np_image = tensor_to_numpy(image_tensor.squeeze(0))

    def batch_predict(images):
        model.eval()
        batch = []
        for img in images:
            preprocess = T.Compose([
                T.ToPILImage(),
                T.Resize((224, 224)),
                T.ToTensor(),
                T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                            std=[0.26862954, 0.26130258, 0.27577711])
            ])
            tensor_img = preprocess(img).to(device)
            batch.append(tensor_img)
        batch_tensor = torch.stack(batch)

        with torch.no_grad():
            outputs = model(batch_tensor)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
        return probs


    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        np_image,
        batch_predict,
        top_labels=5,
        hide_color=0,
        num_samples=1000
    )

    #visualization
    segments = explanation.segments
    weights = dict(explanation.local_exp[pred_class])

    heatmap = np.zeros(segments.shape)
    for sp in np.unique(segments):
        heatmap[segments == sp] = weights.get(sp, 0)

    max_val = np.max(np.abs(heatmap))
    if max_val > 0:
        heatmap = heatmap / max_val

    cmap = cm.seismic
    norm = Normalize(vmin=-1, vmax=1)
    heatmap_color = cmap(norm(heatmap))[:, :, :3]

    # Improved blending
    overlay = 0.4 * heatmap_color + 0.6 * (np_image / 255.0)
    overlay = np.clip(overlay, 0, 1)

    # Add superpixel boundaries
    heatmap_with_boundaries = mark_boundaries((heatmap_color * 255).astype(np.uint8), segments)
    overlay_with_boundaries = mark_boundaries((overlay * 255).astype(np.uint8), segments)

    return {
        "title": f"LIME Explanation",
        "original": np_image,
        "heatmap": heatmap,  # raw grayscale values
        "overlay": overlay_with_boundaries  # improved overlay with boundaries
    }

In [9]:
def get_attr_map(original_img_tensor, attribution_map, title):

    original_img = original_img_tensor.squeeze(0).permute(1, 2, 0).cpu().detach().numpy()
    original_img = (original_img - original_img.min()) / (original_img.max() - original_img.min())  # normalize to [0,1]

    attribution_map_resized = attribution_map
    if attribution_map.shape != original_img.shape[:2]:
        attribution_map_resized = cv2.resize(attribution_map, (original_img.shape[1], original_img.shape[0]))

    heatmap = (attribution_map_resized - np.min(attribution_map_resized)) / (np.max(attribution_map_resized) - np.min(attribution_map_resized) + 1e-6)

    overlay = original_img.copy()
    overlay_map = plt.get_cmap('hot')(heatmap)[..., :3]
    overlay = (0.5 * overlay + 0.5 * overlay_map)

    return {
        "title": title,
        "original": original_img,
        "heatmap": heatmap,
        "overlay": overlay
    }

In [10]:
def visualize_integrated_gradients(input_tensor, model, device, pred_class):
    baseline = torch.zeros_like(input_tensor).to(device)

    ig = IntegratedGradients(model)
    attr, delta = ig.attribute(
        input_tensor,
        baselines=baseline,
        target=pred_class,
        return_convergence_delta=True,
        n_steps=100, #modified
        internal_batch_size=5, #modified
    )

    attributions = attr.squeeze().cpu().detach().numpy()
    attributions = np.sum(attributions, axis=0)
    attributions = np.maximum(attributions, 0)
    attributions = attributions / (attributions.max() + 1e-6)

    return get_attr_map(input_tensor, attributions, "Integrated Gradients")

In [31]:
def visualize_explainability(model_name="first_attempt_with_class_weighting",model_dir="invasive_species",folder_name="taxas",num_images=5,target_label_name=None,
                             species=None,use_bioclip=True,bioclip_version=None,integrated_gradients=True, shap=True, lime=True, output_explanations_dir="explanations"):
    torch.cuda.empty_cache()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_name = f'bioclip_2_{model_name}' if use_bioclip and bioclip_version==2 else f'bioclip_1_{model_name}' if use_bioclip else f'resnet_{model_name}'
    classifier_path = f'/content/drive/MyDrive/TESI - PoliTO/{model_dir}/classifier_head_{model_name}.pth'
    github = '/content/ai4biological-pattern/barbara_new'
    github_repository = f'{github}/{folder_name}'

    if not os.path.isdir(github_repository):
        !git sparse-checkout set barbara_new/{folder_name}
        !git checkout clean-barbara

    feature_extractor, preprocess = get_bioclip_embeddings_extractor_model(device, bioclip_version=bioclip_version)
    feature_extractor.eval().to(device)

    classifier = InvasiveSpeciesClassifier(embedding_dim=get_embedding_dim(feature_extractor, preprocess, device))
    classifier.load_state_dict(torch.load(classifier_path, map_location=device))
    classifier.eval().to(device)

    model = FullModel(feature_extractor, classifier).to(device)
    model.eval()

    dataset = InvasiveSpeciesDataset(root_dir=github_repository, transform=preprocess)
    indices = list(range(len(dataset)))
    random.shuffle(indices)

    label_id = dataset.label_map.get(target_label_name) if target_label_name else None

    processed = 0

    for i in indices:
        if processed >= num_images:
            break
        print(f"Processed {processed} of {num_images}")

        image_tensor, label, idx = dataset[i]

        if label_id is not None and label != label_id:
            print("skip")
            continue

        taxon, row = dataset.entries[idx]
        if species and taxon != species:
            continue

        filename = row["filename"]
        image_path = os.path.join(github_repository, taxon, filename)

        print(f"\nProcessing: {filename} (Species: {taxon})")

        input_tensor = image_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
            confidence = probs[0, pred_class].item()

        print(f"Predicted class: {pred_class}, confidence: {confidence:.4f}")
        if label_id is not None and pred_class != label_id:
            print("mis-classified")
            continue
        figures = []

        if integrated_gradients:
            figures.append(visualize_integrated_gradients(input_tensor, model, device, pred_class))

        if shap:
            figures.append(visualize_shap(input_tensor, model, pred_class, device))

        if lime:
            figures.append(visualize_lime(image_tensor, model, pred_class, device))


        num_methods = len(figures)
        fig, axs = plt.subplots(num_methods, 3, figsize=(18, 6 * num_methods))
        if num_methods == 1:
            axs = axs.reshape(1, -1)
        fig.suptitle(f"Explanations for {filename}", fontsize=18)

        for i, fig_data in enumerate(figures):
            fig.text(0.05, 1 - (i + 0.5) / num_methods, fig_data["title"], fontsize=16, ha='left', va='center', rotation=90)
            axs[i, 0].imshow(fig_data["original"])
            axs[i, 0].set_title("Original Image")
            axs[i, 0].axis('off')

            axs[i, 1].imshow(fig_data["heatmap"])
            axs[i, 1].set_title("Heatmap")
            axs[i, 1].axis('off')

            axs[i, 2].imshow(fig_data["overlay"])
            axs[i, 2].set_title("Overlay")
            axs[i, 2].axis('off')

        plt.tight_layout(rect=[0.1, 0, 1, 0.95])

        if use_bioclip and bioclip_version==2:
          save_path = f'/content/drive/MyDrive/TESI - PoliTO/{output_explanations_dir}/bioclip_2_explanation_{filename}.png'
        else:
          save_path = f'/content/drive/MyDrive/TESI - PoliTO/{output_explanations_dir}/bioclip_explanation_{filename}.png'
        fig.savefig(save_path)
        plt.close(fig)

        print(f"{filename} explanation figure saved..")

        processed += 1

In [29]:
visualize_explainability(bioclip_version=2, num_images=50)

BioCLIP 2!
Model obtained...

Processing: lythrum_tribracteatum_048.jpg (Species: lythrum_tribracteatum)
Predicted class: 0, confidence: 1.0000


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_tribracteatum_048.jpg explanation figure saved..

Processing: lythrum_hyssopifolia_6078.jpeg (Species: lythrum_hyssopifolia)
Predicted class: 2, confidence: 1.0000


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_hyssopifolia_6078.jpeg explanation figure saved..

Processing: lythrum_salicaria_12799.jpeg (Species: lythrum_salicaria)
Predicted class: 0, confidence: 1.0000


...

lythrum_salicaria_2780.jpeg explanation figure saved..

Processing: lythrum_salicaria_8845.jpeg (Species: lythrum_salicaria)
Predicted class: 2, confidence: 1.0000


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_salicaria_8845.jpeg explanation figure saved..

Processing: lythrum_californicum_895.jpg (Species: lythrum_californicum)
Predicted class: 0, confidence: 1.0000


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_californicum_895.jpg explanation figure saved..


In [32]:
visualize_explainability(num_images=50)

open_clip_pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

open_clip_config.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

Model obtained...
Processed 0 of 50

Processing: lythrum_portula_2196.jpeg (Species: lythrum_portula)
Predicted class: 1, confidence: 0.9989


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_portula_2196.jpeg explanation figure saved..
Processed 1 of 50

Processing: lythrum_salicaria_8696.jpeg (Species: lythrum_salicaria)
Predicted class: 2, confidence: 1.0000


...

lythrum_hyssopifolia_4872.jpeg explanation figure saved..
Processed 48 of 50

Processing: lythrum_salicaria_6733.jpeg (Species: lythrum_salicaria)
Predicted class: 0, confidence: 0.9247


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_salicaria_6733.jpeg explanation figure saved..
Processed 49 of 50

Processing: lythrum_hyssopifolia_9059.jpeg (Species: lythrum_hyssopifolia)
Predicted class: 2, confidence: 1.0000


  0%|          | 0/1000 [00:00<?, ?it/s]

lythrum_hyssopifolia_9059.jpeg explanation figure saved..
